In [2]:
import torch
from torch.utils.data import DataLoader
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, AdamW
from peft import PromptTuningConfig, get_peft_model, TaskType
from datasets import load_from_disk
from qwen_vl_utils import process_vision_info

# --- Load model and processor ---
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_name)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [3]:

# Set soft prompt tuning config
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=20,
    tokenizer_name_or_path=model_name,
)

model.word_embeddings = model.get_input_embeddings()
perft_model = get_peft_model(model, peft_config)
perft_model.word_embeddings = model.base_model.get_input_embeddings()

In [9]:

# --- Load and preprocess dataset ---
dataset_path = "/scratch/izar/vanousek/vlm_r1/data/vsr/"
dataset = load_from_disk(dataset_path)['train']  # use 'train' split
dataset = dataset.map(lambda sample: {
    "problem": f'Is the following statement true: {sample["caption"]}',
    "solution": "True" if sample["label"] == 1 else "False"
}, remove_columns=["caption", "label", "relation", "subj", "obj"], desc="Preprocessing")

# --- Format messages for Qwen ---
def make_conversation(example):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{example['image_path']}"},
                {"type": "text", "text": f"{example['problem']} First output the thinking process in <think> </think> tags and then output the final answer in <answer> </answer> tags."},
            ],
        }
    ]
    return {"messages": messages, "solution": example['solution']}

# dataset = dataset.map(make_conversation)

dataset = [make_conversation(sample) for sample in dataset]

# --- Collate function for DataLoader ---
def collate_fn(examples):
    # Apply chat template to text
    texts = [
        processor.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=True)
        for example in examples
    ]

    # Process images (or videos)
    image_inputs = []
    for example in examples:
        imgs, vids = process_vision_info(example["messages"])
        image_inputs.append(imgs)

    # Tokenize multimodal input
    batch = processor(
        text=texts,
        images=image_inputs,
        return_tensors="pt",
        padding=True,
    )

    # Clone input_ids to use as labels
    labels = batch["input_ids"].clone()

    # Mask out pad tokens
    labels[labels == processor.tokenizer.pad_token_id] = -100

    # Mask out special image tokens (they aren't part of the loss)
    image_token_id = processor.tokenizer.convert_tokens_to_ids(processor.image_token)
    labels[labels == image_token_id] = -100

    # 🔥 Mask out soft prompt tokens (first N tokens)
    labels[:, :SOFTPROMPT_LEN] = -100

    batch["labels"] = labels
    return batch

# --- Dataloader ---
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate)


In [10]:
item = next(iter(dataloader))

In [11]:
item

({'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
           151645,    198, 151644,    872,    198, 151652, 151655, 151655, 151655,
           151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
           151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
           151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
           151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
           151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
           151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
           151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
           151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
           151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,
           151655, 151655, 151655, 151655, 151655, 151655, 151655, 151655,

In [14]:
i, out = item
model(**i).logits.shape

torch.Size([2, 408, 151936])

In [15]:
perft_model(**i).logits.shape

torch.Size([2, 428, 151936])

In [16]:
out.shape

torch.Size([2, 1])